In [1]:
import os, torch
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'max_split_size_mb:128'
torch.cuda.empty_cache()

# pretrained model

In [2]:
from model_skingpt4 import *

/home/jq2uw/miniconda3/envs/skingpt4/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
model, vis_processor, chat = init_chat(0, "skingpt_io_eval_llama2_13bchat")
sum(p.numel() for p in model.parameters() if p.requires_grad), sum(p.numel() for p in model.parameters())

Initializing Chat
Loading VIT


/home/jq2uw/miniconda3/envs/skingpt4/lib/python3.9/site-packages/huggingface_hub/file_download.py:945: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading VIT Done
Loading Q-Former
Loading Q-Former Done
Loading LLM tokenizer
Loading LLM model


Loading checkpoint shards: 100%|███████████████████████████████████████████████████| 3/3 [00:03<00:00,  1.26s/it]


Loading LLM Done
Load 2 training prompts
Prompt Example 
###Human: <Img><ImageHere></Img> Could you describe the skin disease in this image for me? ###Assistant: 
Load BLIP2-LLM Checkpoint: /scratch/jq2uw/edit-skingpt4/model_skingpt4/weights/skingpt4_llama2_13bchat_base_pretrain_stage2.pth
Initialization Finished


(109099520, 14110861184)

In [4]:
print(type(model).__name__) 

skingpt_io


# data

In [5]:
# import os
# os.chdir("/scratch/jq2uw/edit-skingpt4")

In [6]:
from data_utils import *

In [7]:
df = process_tabular("./data")
train_df, val_df, test_df = split_df(df)
train_dataset = MIDASDataset(train_df, "./data")
val_dataset = MIDASDataset(val_df, "./data")
test_dataset = MIDASDataset(test_df, "./data")

id_patient: 733
id_filename: 3416
midas_path: 17
midas_path: {'malignant-bcc': 608, 'benign-melanocytic nevus': 578, 'nan': 497, 'benign-other': 421, 'benign-seborrheic keratosis': 242, 'malignant-melanoma': 238, 'malignant-scc': 203, 'malignant-ak': 191, 'malignant-sccis': 165, 'other-melanocytic lesion, possible re-excision (severe, spitz, aimp)': 109, 'benign-dermatofibroma': 51, 'other-non-neoplastic, inflammatory, infectious': 39, 'benign-hemangioma': 30, 'benign-fibrous papule': 18, 'malignant-other': 14, 'melanocytic tumor, possible re-excision (severe, spitz, aimp)': 6, 'unknown': 6}
y16_description: 16
y16: 16
y16: {'Basal Cell Carcinoma': 608, 'Melanocytic Nevus': 578, 'Other Benign': 421, 'Seborrheic Keratosis': 242, 'Melanoma': 238, 'Squamous Cell Carcinoma': 203, 'Actinic Keratosis': 191, 'Squamous Cell Carcinoma In Situ': 165, 'Melanocytic Lesion': 109, 'Dermatofibroma': 51, 'Non-neoplastic': 39, 'Hemangioma': 30, 'Fibrous Papule': 18, 'Other Malignant': 14, 'Melanocytic 

## finetune

In [14]:
from finetune_utils import *
# vis_processor is already initialized from run/init.py
TARGET = 'text_full'
train_ds_ft = MIDASFTSkGPTIODataset(train_dataset, vis_processor, prompt_keys=["text_demo"], answer_keys=['y3'])
val_ds_ft   = MIDASFTSkGPTIODataset(val_dataset,   vis_processor, prompt_keys=["text_demo"], answer_keys=['y3'])
train_loader = train_ds_ft.get_loader(batch_size=2, shuffle=True, num_workers=2)
val_loader   = val_ds_ft.get_loader(batch_size=2, shuffle=False, num_workers=2)


In [15]:
model.finetune(train_loader, val_loader, n_epochs=10, retrain=True,
        lr=1e-4, weight_decay=0.5, ckpt_path=f"./model_skingpt4/weights/finetune_skingpt_io_{TARGET}.pth")

epoch 1/10  train_loss=0.2786  val_loss=0.2596

KeyboardInterrupt: saved checkpoint to ./model_skingpt4/weights/finetune_skingpt_io_text_full.pth


## eval

In [16]:
from eval_utils import *

In [18]:
# one example
i = 16
image = test_dataset[i]['image']
print(f"ground truth: {test_dataset[i]['y']['y3']}")
print("-" * 50)
print("Pretrained model")
model = load_model_weights(model, "./model_skingpt4/weights/skingpt4_llama2_13bchat_base_pretrain_stage2.pth")
resp = chat_with_image(chat, image, "Is the lesion malignant or benign, or unknown?", temperature=0.01)
print(resp)
print("-" * 50)
print(f"Finetuned model (y3)")
model = load_model_weights(model, "./model_skingpt4/weights/finetune_llama.pth")
resp = chat_with_image(chat, image, "Is the lesion malignant or benign, or unknown?", temperature=0.01)
print(resp)
print(f"Finetuned model ({TARGET})")
model = load_model_weights(model, f"./model_skingpt4/weights/finetune_skingpt4_{TARGET}.pth")
resp = chat_with_image(chat, image, "Is the lesion malignant or benign, or unknown?", temperature=0.01)
print(resp)

ground truth: malignant
--------------------------------------------------
Pretrained model
malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignan

In [20]:

print("-" * 50)
print(f"Finetuned io model ({TARGET})")
model = load_model_weights(model, f"./model_skingpt4/weights/finetune_skingpt_io_{TARGET}.pth")
resp = chat_with_image(chat, image, 
      f"{test_dataset[i]['y']['text_demo']}. Is the lesion malignant or benign, or unknown?",
      temperature=0.05)
print(resp)

--------------------------------------------------
Finetuned io model (text_full)
malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignant malignan

In [28]:
# random 200 unique samples (reproducible)
import random
from torch.utils.data import Subset

random.seed(42)  # optional for determinism
idxs = random.sample(range(len(train_dataset)), k=min(50, len(train_dataset)))
subset = Subset(train_dataset, idxs)

res = eval_ft_skingpt4(chat, subset, 
                       temperature=0.01, target=TARGET, 
                       question=None)
res

  2%|█▌                                                                           | 1/50 [00:28<23:01, 28.18s/it]


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:9                                                                                    │
│                                                                                                  │
│    6 idxs = random.sample(range(len(train_dataset)), k=min(50, len(train_dataset)))              │
│    7 subset = Subset(train_dataset, idxs)                                                        │
│    8                                                                                             │
│ ❱  9 res = eval_ft_skingpt4(chat, subset,                                                        │
│   10 │   │   │   │   │      temperature=0.01, target=TARGET,                                     │
│   11 │   │   │   │   │      question=None)                                                       │
│   12 res                                                                                         │
│                                                                                                  │
│ in eval_ft_skingpt4:91                                                                           │
│                                                                                                  │
│    88 │   │   gt = _normalize(sample['y'][target], target)                                       │
│    89 │   │   if question is None:                                                               │
│    90 │   │   │   question = f"{sample['y']['text_demo']}. Is the lesion malignant or benign,    │
│ ❱  91 │   │   pred = _normalize(chat_with_image(chat, img, question, temperature=temperature),   │
│    92 │   │   y_true.append(gt)                                                                  │
│    93 │   │   y_pred.append(pred)                                                                │
│    94 │   │   rows.append({                                                                      │
│                                                                                                  │
│ /sfs/weka/scratch/jq2uw/edit-skingpt4/model_skingpt4/__init__.py:83 in chat_with_image           │
│                                                                                                  │
│   80 │   img_list = []                                                                           │
│   81 │   _ = chat.upload_img(image, chat_state, img_list)                                        │
│   82 │   chat.ask(question, chat_state)                                                          │
│ ❱ 83 │   response = chat.answer(                                                                 │
│   84 │   │   conv=chat_state,                                                                    │
│   85 │   │   img_list=img_list,                                                                  │
│   86 │   │   num_beams=num_beams,                                                                │
│                                                                                                  │
│ /sfs/weka/scratch/jq2uw/edit-skingpt4/model_skingpt4/skingpt4/conversation/conversation.py:150   │
│ in answer                                                                                        │
│                                                                                                  │
│   147 │   │                                                                                      │
│   148 │   │   embs = embs[:, begin_idx:]                                                         │
│   149 │   │                                                                                      │
│ ❱ 150 │   │   outputs = self.model.llm_model.generate(                                           │
│   151 │   │   │   inputs_embeds=embs,                                                            │
│   152 │   │   │   max_new_tokens=max_new_tokens,           

In [ ]:
# random 200 unique samples (reproducible)
import random
from torch.utils.data import Subset

random.seed(42)  # optional for determinism
idxs = random.sample(range(len(test_dataset)), k=min(100, len(test_dataset)))
subset = Subset(test_dataset, idxs)

res = eval_ft_skingpt4(chat, subset, 
                       temperature=0.05, target=TARGET, 
                       question=None)
res

 35%|██████████████████████████▎                                                | 35/100 [08:03<14:58, 13.82s/it]


In [ ]:

# --- eval ---
from eval_utils import *
import os, torch

target = "y3"
question = "Is the lesion malignant or benign, or other?"
res_dir = f"./results/ft_skingpt4_{target}"
os.makedirs(res_dir, exist_ok=True)
for split_name, ds in [("test", test_dataset), ("train", train_dataset), ("val", val_dataset)]:
    res_fname = f"{res_dir}/eval_{split_name}.pth"  
    if os.path.exists(res_fname):
        continue
    res = eval_ft_skingpt4(chat, ds, temperature=0.01, target=target, question=question)
    torch.save(res, res_fname)

 68%|████████████████████████████████████████████████▋                       | 243/359 [02:19<01:14,  1.56it/s]

: 

In [ ]:
# --- eval ---
import os, torch
from pprint import pprint
target = "y3"
question = "Is the lesion malignant or benign, or other?"
res_dir = f"../results/ft_skingpt4_{target}"
for split_name in ["test", "train"]:
    print(split_name)
    res = torch.load(f"{res_dir}/eval_{split_name}.pth")
    pprint(res)


test
{'accuracy': 0.6120689655172413,
 'confusion': array([[ 73, 118,  42],
       [ 28, 289,  22],
       [ 26,  34,  64]]),
 'f1': 0.5515059015059015,
 'precision': 0.5767106492640801,
 'recall': 0.5606470426397919,
 'report': '              precision    recall  f1-score   support\n'
           '\n'
           '      benign       0.57      0.31      0.41       233\n'
           '   malignant       0.66      0.85      0.74       339\n'
           '       other       0.50      0.52      0.51       124\n'
           '\n'
           '    accuracy                           0.61       696\n'
           '   macro avg       0.58      0.56      0.55       696\n'
           'weighted avg       0.60      0.61      0.59       696\n'}
train
{'accuracy': 0.7576020851433536,
 'confusion': array([[519, 279, 130],
       [ 25, 877,  28],
       [ 19,  74, 348]]),
 'f1': 0.7469485594528837,
 'precision': 0.7742008041820251,
 'recall': 0.7620205926170888,
 'report': '              precision    recall  